In [17]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly.express as px
import logging
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

mpl.rcParams['figure.dpi'] = 150
plt.style.use('seaborn-v0_8-whitegrid')
logging.disable(logging.CRITICAL)

In [18]:
df = pd.read_csv('../data/eu_regional_data.csv', index_col = 'region_name')
df.head()

,Country,EU Region,GDP,GDP per Capita,Unemployment %,Life Expectancy,Doctors per 100K,Heart Disease Deaths per 100K,Cancer Deaths per 100K,Fatal Road Accidents per Million,Tertiary Educational Attainment %,Population Density,People at Risk of Poverty %,Regular Internet Users %
region_name,,,,,,,,,,,,,,
Région de Bruxelles-Capitale/ Brussels Hoofdstedelijk Gewest,BE,Western Europe,97.61,78700.0,11.4,81.6,399.54,51.30,210.99,20.0,53.5,7660.0,38.8,89.91
Prov. Antwerpen,BE,Western Europe,109.40,57500.0,4.0,82.8,278.84,45.29,208.76,34.0,47.1,678.8,14.0,90.24
Prov. Limburg (BE),BE,Western Europe,34.91,39100.0,3.5,82.8,267.07,41.98,205.30,56.0,43.7,373.8,11.5,90.62
Prov. Oost-Vlaanderen,BE,Western Europe,71.18,45800.0,2.0,82.6,304.81,45.68,215.82,39.0,49.6,523.4,10.1,88.97
Prov. Vlaams-Brabant,BE,Western Europe,64.45,54600.0,3.6,83.2,414.90,44.40,204.79,33.0,51.6,558.7,9.5,90.43


In [19]:
df.isna().sum()

Country                               0
EU Region                             0
GDP                                   0
GDP per Capita                        0
Unemployment %                        6
Life Expectancy                       0
Doctors per 100K                     65
Heart Disease Deaths per 100K         0
Cancer Deaths per 100K                0
Fatal Road Accidents per Million      1
Tertiary Educational Attainment %     1
Population Density                    2
People at Risk of Poverty %           8
Regular Internet Users %             73
dtype: int64

In [20]:
cols_num = [ 'GDP', 'GDP per Capita', 'Unemployment %',
       'Life Expectancy', 'Doctors per 100K', 'Heart Disease Deaths per 100K',
       'Cancer Deaths per 100K', 'Fatal Road Accidents per Million',
       'Tertiary Educational Attainment %', 'Population Density',
       'People at Risk of Poverty %', 'Regular Internet Users %']

imputer = KNNImputer()
df[cols_num] = imputer.fit_transform(df[cols_num])


In [21]:
# scaling variables
scaler = StandardScaler()
X = scaler.fit_transform(df[cols_num])
#X = df[cols_num]

In [22]:
# Principal Component Analysis
pca = PCA(n_components = 2)
components = pca.fit_transform(X)
total_var = pca.explained_variance_ratio_.sum() * 100

df_pca = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))
df_pca.reset_index(inplace = True)

fig = px.scatter(df_pca, x = 'pc_1', y = 'pc_2', color=df_pca['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 hover_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



In [23]:
# t-SNE
tsne = TSNE(n_components = 2, perplexity = 100, random_state=42)
components = tsne.fit_transform(X)

df_tsne = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))

df_tsne.reset_index(inplace = True)

fig = px.scatter(df_tsne, x = 'pc_1', y = 'pc_2', color=df_tsne['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 hover_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



In [24]:
# UMAP
umap_model = umap.UMAP(n_components = 2, n_neighbors = 25,
                       random_state = 42)
components = umap_model.fit_transform(X)

df_umap = df.join(pd.DataFrame(components, index = df.index,
                           columns = ['pc_1', 'pc_2']))

df_umap.reset_index(inplace = True)

fig = px.scatter(df_umap, x = 'pc_1', y = 'pc_2', color=df_umap['EU Region'],
                 width = 800, height = 600, size = 'GDP per Capita',
                 custom_data = ['region_name', 'GDP per Capita'])

fig.update_layout(  margin={"r":1,"t":15,"l":1,"b":1},
                    plot_bgcolor = '#FFFFFF',
                    legend = dict(orientation = 'h', yanchor = 'top'),
                    yaxis_title='', xaxis_title='',
                    title = "")

hovertemplate = '%{customdata[0]}' 
fig.update_traces(hovertemplate=hovertemplate)
fig.show()



c:\Users\jtoli\miniconda3\envs\sktime\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

c:\Users\jtoli\miniconda3\envs\sktime\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [25]:
#saving datasets 
df_pca.to_csv('../data/eu_regional_data_pca.csv',
          float_format = '%.2f', encoding = 'utf-8')

df_tsne.to_csv('../data/eu_regional_data_tsne.csv',
          float_format = '%.2f', encoding = 'utf-8')

df_umap.to_csv('../data/eu_regional_data_umap.csv',
          float_format = '%.2f', encoding = 'utf-8')